In [15]:
import torch
from encodec import EncodecModel

# Load a pretrained model
models = {
        'encodec_24khz': EncodecModel.encodec_model_24khz,
        'encodec_48khz': EncodecModel.encodec_model_48khz
}

bandwidths = [3, 6, 12, 24]



In [16]:
model_name = 'encodec_24khz'
model = models[model_name]()
model.set_target_bandwidth(bandwidths[2])  # Set to a specific bandwidth
model.exporting_to_onnx = True

/Users/bezha/anaconda3/lib/python3.12/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


In [17]:
import torchaudio
audio_suffix = model_name.split('_')[1][:3]
wav, sr = torchaudio.load(f"test_{audio_suffix}.wav")

In [18]:
wav = wav[:, :model.sample_rate * 2]
print(wav.shape)
wav_in = wav.unsqueeze(0)
print(wav_in.shape)
wav_rec = model.forward(wav_in)
print(wav_rec.shape)

torch.Size([1, 48000])
torch.Size([1, 1, 48000])
segment_length:  None
torch.Size([1, 1, 48000])


In [19]:
# encode
encoded_frames = model.encode(wav_in)
encodings = [ef[0] for ef in encoded_frames]
scaling_factors = [ef[1] for ef in encoded_frames]

print("Is Normalized: ", model.normalize)
print("n frames: ", len(encoded_frames))
print("Batch size: ", encodings[0].shape[0])
print("Scaling factors: ", scaling_factors)
print("Codecs: ", [e.shape for e in encodings])

segment_length:  None
Is Normalized:  False
n frames:  1
Batch size:  1
Scaling factors:  [None]
Codecs:  [torch.Size([1, 16, 150])]


In [20]:
# decode
decoded_frames = model.decode(encoded_frames)[:, :, :wav_in.shape[-1]]
decoded_frames.shape

torch.Size([1, 1, 48000])

In [21]:
model.forward(wav_in).shape

segment_length:  None


torch.Size([1, 1, 48000])

In [22]:
class EncodecEncoderWrapper(torch.nn.Module):
    def __init__(self, encodec_model):
        super(EncodecEncoderWrapper, self).__init__()
        self.encodec_model = encodec_model

    def forward(self, x):
        # assume no batch dimension so it should be (n_channels, n_samples)
        encoded_frames = self.encodec_model.encode(x.unsqueeze(0))
        return encoded_frames

encoder_wrapper = EncodecEncoderWrapper(model)
print(wav_in[0].shape)
torch.onnx.export(
    encoder_wrapper,
    wav_in[0],
    "encodec_encoder.onnx",
    opset_version=13,
    input_names=["audio_input"],
    output_names=["encoded_frames"],
    dynamic_axes={"audio_input": {1: "n_samples"}}
)

torch.Size([1, 48000])
segment_length:  None


/var/folders/lr/8ctpqx7n6m54ydpt525nf6q80000gn/T/ipykernel_12734/4071034345.py:13: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter will be the default. To switch now, set dynamo=True in torch.onnx.export. This new exporter supports features like exporting LLMs with DynamicCache. We encourage you to try it and share feedback to help improve the experience. Learn more about the new export logic: https://pytorch.org/docs/stable/onnx_dynamo.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html.
  torch.onnx.export(


In [23]:
import torch

class EncodecEncoderWrapper(torch.nn.Module):
    def __init__(self, encodec_model):
        super(EncodecEncoderWrapper, self).__init__()
        self.encodec_model = encodec_model

    def forward(self, x):
        # x is [n_channels, n_samples]
        # Add batch dimension -> [1, n_channels, n_samples]
        x_batched = x.unsqueeze(0)

        # encode() returns List[EncodedFrame] where EncodedFrame = (codes, scale)
        encoded_frames = self.encodec_model.encode(x_batched)

        # Extract just the codes from the first (and only) frame
        codes = encoded_frames[0][0]  # Shape: [B, K, T] = [1, n_codebooks, n_frames]

        # Remove batch dimension for output -> [n_codebooks, n_frames]
        codes = codes.squeeze(0)
        print(codes.shape)

        return codes

# Setup
model = EncodecModel.encodec_model_48khz()
model.set_target_bandwidth(3.0)  # Set bandwidth first!
model.eval()

encoder_wrapper = EncodecEncoderWrapper(model)

# Create dummy input: [n_channels, n_samples]
dummy_input = torch.randn(2, 48000)  # 1 second of 48kHz stereo audio

print("Input shape:", dummy_input.shape)

# Test forward pass
with torch.no_grad():
    output = encoder_wrapper(dummy_input)
    print("Output shape:", output.shape)  # Should be [n_codebooks, n_frames]

# Export to ONNX
torch.onnx.export(
    encoder_wrapper,
    dummy_input,
    "encodec_encoder.onnx",
    opset_version=13,
    input_names=["audio_input"],
    output_names=["encoded_codes"],
    dynamic_axes={
        "audio_input": {1: "n_samples"},      # Variable length audio
        "encoded_codes": {1: "n_frames"}      # Variable number of frames
    }
)

print("✅ ONNX export complete!")

Input shape: torch.Size([2, 48000])
segment_length:  48000
torch.Size([2, 150])
Output shape: torch.Size([2, 150])
segment_length:  48000


/var/folders/lr/8ctpqx7n6m54ydpt525nf6q80000gn/T/ipykernel_12734/3144741878.py:43: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter will be the default. To switch now, set dynamo=True in torch.onnx.export. This new exporter supports features like exporting LLMs with DynamicCache. We encourage you to try it and share feedback to help improve the experience. Learn more about the new export logic: https://pytorch.org/docs/stable/onnx_dynamo.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html.
  torch.onnx.export(


torch.Size([2, 150])
✅ ONNX export complete!


In [10]:
import onnx
model = onnx.load("encodec_encoder.onnx")
print("Input shape:", model.graph.input[0].type.tensor_type.shape)

Input shape: dim {
  dim_value: 2
}
dim {
  dim_param: "n_samples"
}



In [19]:
model.segment_stride, model.segment_length

(47520, 48000)

In [20]:
class EncodecDecoderWrapper(torch.nn.Module):
    def __init__(self, encodec_model):
        super(EncodecDecoderWrapper, self).__init__()
        self.encodec_model = encodec_model

    def forward(self, encoded_frames):
        # assume no batch dimension so it should be (n_channels, n_samples)
        decoded_audio = self.encodec_model.decode(encoded_frames)[0]
        return decoded_audio

decoder_wrapper = EncodecDecoderWrapper(model)

torch.onnx.export(
    decoder_wrapper,
    encoded_frames,
    "encodec_decoder.onnx",
    opset_version=13,
    input_names=["encoded_frames"],
    output_names=["decoded_audio"],
    dynamic_axes={"encoded_frames": {2: "n_frames"}},
)


/var/folders/lr/8ctpqx7n6m54ydpt525nf6q80000gn/T/ipykernel_58231/3375827038.py:13: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter will be the default. To switch now, set dynamo=True in torch.onnx.export. This new exporter supports features like exporting LLMs with DynamicCache. We encourage you to try it and share feedback to help improve the experience. Learn more about the new export logic: https://pytorch.org/docs/stable/onnx_dynamo.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html.
  torch.onnx.export(
